In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

import os
import time
import json

import numpy as np
import time
import nibabel as nib
import os
import json
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from torch.optim import AdamW
from scipy.ndimage import distance_transform_edt
import time
from typing import Dict, Optional, List, Union
from timm.optim.lookahead import Lookahead

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [ ]:

from google.colab import drive

drive.mount('/content/drive') # mount drive


"""
"tensorImageSize": "4D",
"modality": {
	 "0": "FLAIR",
	 "1": "T1w",
	 "2": "t1gd",
	 "3": "T2w"
 },
 "labels": {
	 "0": "background",
	 "1": "edema",
	 "2": "non-enhancing tumor",
	 "3": "enhancing tumour"
 },
"""

braintumor_datadir = "/content/drive/MyDrive/data/med/braintumor0"

bt_train_images_dir = os.path.join(braintumor_datadir, "imagesTr")
bt_train_labels_dir = os.path.join(braintumor_datadir, "labelsTr")

Mounted at /content/drive


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
import nibabel as nib
import os

def percentile_normalization_numpy(data_np, modality, stats_dict):
    """
    Normalize using GLOBAL percentiles, preserving background as 0
    - Background (≤p0.5): stays 0
    - Foreground (p0.5 to p99.5): maps to (0,1)
    - Outliers (>p99.5): clipped to 1
    """
    params = stats_dict[modality]
    p_low, p_high = params['p0.5'], params['p99.5']
    range_ = p_high - p_low

    # Initialize with zeros (background stays 0)
    normalized = np.zeros_like(data_np, dtype=np.float32)

    # Only normalize foreground voxels (above noise threshold)
    foreground_mask = data_np > p_low

    if np.any(foreground_mask):
        foreground_values = data_np[foreground_mask]
        # Map [p_low, p_high] → [0, 1], clip outliers
        scaled = (foreground_values - p_low) / range_
        scaled = np.clip(scaled, 0, 1)
        normalized[foreground_mask] = scaled

    return normalized

def z_score_normalization(x: torch.Tensor, eps=1e-8) -> torch.Tensor:
    """Vectorized per-channel Z-score normalization"""
    # Compute mean and std across spatial dimensions (1,2,3), keep channel dim
    mean = x.mean(dim=(1,2,3), keepdim=True)
    std = x.std(dim=(1,2,3), keepdim=True) + eps
    return (x - mean) / std

class BratsDataset(Dataset):
    def __init__(self, images_dir, labels_dir, global_stats=None):
        super().__init__()
        self.images_dir = images_dir
        self.labels_dir = labels_dir

        # Get all .nii.gz files
        self.image_files = {f for f in os.listdir(images_dir) if f.endswith('.nii.gz')}
        self.label_files = {f for f in os.listdir(labels_dir) if f.endswith('.nii.gz')}
        self.common_files = sorted(list(self.image_files & self.label_files))

        if len(self.common_files) == 0:
            raise ValueError("No matching files found between images_dir and labels_dir")

        self.modalities = ['FLAIR', 'T1w', 'T1GD', 'T2w']

        # Use provided global stats or defaults
        if global_stats is not None:
            self.global_stats = global_stats
        else:
            # Your precomputed GLOBAL stats across entire dataset
            self.global_stats = {
                'FLAIR': {'p0.5': 23.958, 'p99.5': 1168.326},
                'T1w': {'p0.5': 45.15, 'p99.5': 2012.474},
                'T1GD': {'p0.5': 38.086, 'p99.5': 2305.63},
                'T2w': {'p0.5': 41.618, 'p99.5': 1669.87}
            }

        print(f"BratsDataset initialized with {len(self.common_files)} samples")
        print("Using robust percentile normalization:")
        print("- Background voxels (≤p0.5): preserved as 0")
        print("- Foreground (p0.5-p99.5): normalized to [0,1]")
        print("- Outliers (>p99.5): clipped to 1")

    def __len__(self):
        return len(self.common_files)

    def __getitem__(self, index):
        filename = self.common_files[index]

        try:
            # Load NIfTI files - shape should be (240, 240, 155, 4) and (240, 240, 155)
            image_path = os.path.join(self.images_dir, filename)
            label_path = os.path.join(self.labels_dir, filename)

            image_np = nib.load(image_path).get_fdata().astype(np.float32)  # (240, 240, 155, 4)
            label_np = nib.load(label_path).get_fdata().astype(np.int64)   # (240, 240, 155)

            # Validate shapes
            expected_img_shape = (240, 240, 155, 4)
            expected_lbl_shape = (240, 240, 155)

            if image_np.shape != expected_img_shape:
                raise ValueError(f"Image shape {image_np.shape} != expected {expected_img_shape}")
            if label_np.shape != expected_lbl_shape:
                raise ValueError(f"Label shape {label_np.shape} != expected {expected_lbl_shape}")

            # Apply GLOBAL normalization to each modality
            normalized_channels = []
            for c, modality in enumerate(self.modalities):
                channel_data = image_np[..., c]  # (240, 240, 155)
                normalized_channel = percentile_normalization_numpy(
                    channel_data, modality, self.global_stats
                )
                normalized_channels.append(normalized_channel)

            # Stack and convert to tensor: (240, 240, 155, 4) -> (4, 155, 240, 240)
            image_normalized = np.stack(normalized_channels, axis=-1)  # (240, 240, 155, 4)
            image_tensor = torch.from_numpy(image_normalized).permute(3, 2, 0, 1).float()  # (4, 155, 240, 240)

            # Process label: (240, 240, 155) -> (155, 240, 240)
            label_tensor = torch.from_numpy(label_np).permute(2, 0, 1).long()  # (155, 240, 240)

            # Validate output shapes
            assert image_tensor.shape == (4, 155, 240, 240), f"Wrong image tensor shape: {image_tensor.shape}"
            assert label_tensor.shape == (155, 240, 240), f"Wrong label tensor shape: {label_tensor.shape}"

            # Validate value ranges
            assert torch.all(image_tensor >= 0) and torch.all(image_tensor <= 1), "Image values not in [0,1]"
            assert torch.all((label_tensor >= 0) & (label_tensor <= 3)), "Label values not in {0,1,2,3}"

            return image_tensor, label_tensor

        except Exception as e:
            print(f"Error loading {filename}: {str(e)}")
            raise

    def get_sample_info(self, index):
        """Debug method to inspect sample statistics"""
        filename = self.common_files[index]
        image_tensor, label_tensor = self[index]

        info = {
            'filename': filename,
            'image_shape': image_tensor.shape,
            'label_shape': label_tensor.shape,
            'image_range': (image_tensor.min().item(), image_tensor.max().item()),
            'label_classes': torch.unique(label_tensor).tolist(),
            'channel_stats': {}
        }

        for c, modality in enumerate(self.modalities):
            channel_data = image_tensor[c]
            info['channel_stats'][modality] = {
                'min': channel_data.min().item(),
                'max': channel_data.max().item(),
                'mean': channel_data.mean().item(),
                'std': channel_data.std().item()
            }

        return info
import torch
import torch.nn as nn

class DepthwiseSeparableConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(
            in_channels, in_channels, kernel_size,
            padding=padding, groups=in_channels
        )
        self.pointwise = nn.Conv3d(in_channels, out_channels, 1)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

class AttentionGate3d(nn.Module):
    def __init__(self, F_g, F_l):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv3d(F_g, F_l, kernel_size=1),
            nn.BatchNorm3d(F_l)
        )
        self.W_x = nn.Sequential(
            nn.Conv3d(F_l, F_l, kernel_size=1),
            nn.BatchNorm3d(F_l)
        )
        self.psi = nn.Sequential(
            nn.Conv3d(F_l, 1, kernel_size=1),
            nn.BatchNorm3d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class SparseMedNet3D(nn.Module):
    def __init__(self, in_channels=4, out_channels=1, base_filters=24):
        super().__init__()
        filters = [base_filters, base_filters*2, base_filters*4, base_filters*8]

        # Encoder
        self.enc1 = self.conv_block(in_channels, filters[0])
        self.enc2 = self.conv_block(filters[0], filters[1])
        self.enc3 = self.conv_block(filters[1], filters[2])

        # Bottleneck with dilated convolution
        self.bottleneck = nn.Sequential(
            DepthwiseSeparableConv3d(filters[2], filters[3]),
            nn.BatchNorm3d(filters[3]),
            nn.ReLU(inplace=True),
            DepthwiseSeparableConv3d(filters[3], filters[3]),
            nn.BatchNorm3d(filters[3]),
            nn.ReLU(inplace=True),
            nn.Conv3d(filters[3], filters[3], kernel_size=3, dilation=2, padding=2)
        )

        # Attention gates
        self.attn3 = AttentionGate3d(filters[3], filters[2])
        self.attn2 = AttentionGate3d(filters[2], filters[1])
        self.attn1 = AttentionGate3d(filters[1], filters[0])

        # Decoder
        self.dec3 = self.conv_block(filters[3]+filters[2], filters[2])
        self.dec2 = self.conv_block(filters[2]+filters[1], filters[1])
        self.dec1 = self.conv_block(filters[1]+filters[0], filters[0])

        # Upsample and output
        self.upsample = nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True)
        self.final_conv = nn.Conv3d(filters[0], out_channels, kernel_size=1)

        # Pooling
        self.pool = nn.MaxPool3d(2)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            DepthwiseSeparableConv3d(in_channels, out_channels),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            DepthwiseSeparableConv3d(out_channels, out_channels),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Encoder path
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        # Bottleneck
        b = self.bottleneck(self.pool(e3))

        # Decoder path with attention
        d3 = self.upsample(b)
        a3 = self.attn3(d3, e3)
        d3 = torch.cat([d3, a3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.upsample(d3)
        a2 = self.attn2(d2, e2)
        d2 = torch.cat([d2, a2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.upsample(d2)
        a1 = self.attn1(d1, e1)
        d1 = torch.cat([d1, a1], dim=1)
        d1 = self.dec1(d1)

        return self.final_conv(d1)

        import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedDiceLoss(nn.Module):
    def __init__(self, num_classes=4,
                 false_positive_weights=None, false_negative_weights=None,
                 class_importance_weights=None, epsilon=1e-6):
        """
        Optimized for medical segmentation with shape (B, 4, 155, 240, 240)

        Args:
            num_classes (int): number of segmentation classes (4 for BraTS)
            false_positive_weights (Tensor): shape [4], λ^FP_c per class
            false_negative_weights (Tensor): shape [4], λ^FN_c per class
            class_importance_weights (Tensor, optional): shape [4], γ_c or w_c
            epsilon (float): smoothing to avoid division by zero
        """
        super().__init__()
        self.num_classes = num_classes

        # Default weights optimized for medical segmentation (penalize FN more for rare classes)
        if false_positive_weights is None:
            false_positive_weights = torch.tensor([0.5, 1.0, 1.0, 1.0])  # Less penalty for background FP
        if false_negative_weights is None:
            false_negative_weights = torch.tensor([0.5, 1.5, 2.0, 2.0])  # Higher penalty for tumor FN

        # Register weights as buffers so they are moved to the device with the module
        self.register_buffer('fp_weights', false_positive_weights)
        self.register_buffer('fn_weights', false_negative_weights)
        self.register_buffer('class_weights', class_importance_weights)
        self.epsilon = epsilon

    def forward(self, logits, targets):
        """
        Args:
            logits: Tensor of shape [B, 4, 155, 240, 240]
            targets: Tensor of shape [B, 155, 240, 240] with values in {0,1,2,3}
        Returns:
            Scalar loss value and stats dict
        """
        B, C = logits.shape[:2]
        assert C == 4, f"Expected 4 classes, got {C}"

        probs = F.softmax(logits, dim=1)  # [B, 4, 155, 240, 240]

        # Convert targets to one-hot: [B, 155, 240, 240] -> [B, 4, 155, 240, 240]
        targets_onehot = F.one_hot(targets.long(), num_classes=4).float()
        targets_onehot = targets_onehot.permute(0, 4, 1, 2, 3)  # [B, 4, 155, 240, 240]

        # Compute TP, FP, FN - sum over batch and spatial dims [155, 240, 240]
        TP = (probs * targets_onehot).sum(dim=(0, 2, 3, 4))  # [4]
        FP = (probs * (1 - targets_onehot)).sum(dim=(0, 2, 3, 4))  # [4]
        FN = ((1 - probs) * targets_onehot).sum(dim=(0, 2, 3, 4))  # [4]

        # Compute class frequency weights if not provided
        if self.class_weights is None:
            class_counts = targets_onehot.sum(dim=(0, 2, 3, 4))  # [4]
            class_weights = 1.0 / (class_counts + self.epsilon)
            class_weights = class_weights / class_weights.sum()
        else:
            class_weights = self.class_weights

        # Compute Tversky loss per class
        numerator = TP
        denominator = TP + self.fp_weights * FP + self.fn_weights * FN + self.epsilon
        class_losses = 1.0 - numerator / denominator  # [4]

        # Weighted sum of losses
        total_loss = (class_weights * class_losses).sum()

        stats = {
            'TP': TP.detach().cpu(),
            'FP': FP.detach().cpu(),
            'FN': FN.detach().cpu(),
            'class_losses': class_losses.detach().cpu(),
            'class_weights': class_weights.detach().cpu(),
            'dice_scores': (2 * TP / (2 * TP + FP + FN + self.epsilon)).detach().cpu()
        }

        return total_loss, stats

        import torch
import torch.nn as nn
from torch.optim import AdamW
import os
import time
import json

# FIXED: Proper parameter names for WeightedDiceLoss
def setup_training():
    """Setup optimizer and loss function with corrected parameters"""

    # Initialize model (assuming you have this)
    # model = SparseMedNet3D(in_channels=4, out_channels=4, base_filters=24)

    # Optimizer setup
    optimizer_w_dicev1 = AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-3
    )

    # FIXED: Use correct parameter names from WeightedDiceLoss.__init__
    loss_dice_v1 = WeightedDiceLoss(
        num_classes=4,
        false_positive_weights=torch.tensor([0.1, 1.0, 1.0, 1.0]),  # Less penalty for background FP
        false_negative_weights=torch.tensor([1.0, 1.0, 1.0, 1.0]),  # Standard FN penalty
        class_importance_weights=torch.tensor([0.2, 1.0, 1.0, 1.0])  # Lower weight for background
    )

    return optimizer_w_dicev1, loss_dice_v1

def compute_metrics_from_stats(stats):
    """Compute precision, recall, dice from TP/FP/FN stats"""
    TP = stats['TP'].cpu().numpy()
    FP = stats['FP'].cpu().numpy()
    FN = stats['FN'].cpu().numpy()

    class_metrics = {}
    for i in range(len(TP)):
        # Avoid division by zero
        precision = TP[i] / (TP[i] + FP[i] + 1e-8)
        recall = TP[i] / (TP[i] + FN[i] + 1e-8)
        dice = 2 * TP[i] / (2 * TP[i] + FP[i] + FN[i] + 1e-8)

        class_metrics[f'class_{i}'] = {
            'TP': int(TP[i]),
            'FP': int(FP[i]),
            'FN': int(FN[i]),
            'precision': float(precision),
            'recall': float(recall),
            'dice': float(dice)
        }

    return class_metrics

def trainV1(model, train_loader, loss_fn, optimizer, num_iterations, save_dir,
          device='cuda', early_stop_patience=10, accumulation_steps=4):
    """
    Fixed training function with proper gradient accumulation and scheduling
    """

    os.makedirs(save_dir, exist_ok=True)
    model.to(device)

    # FIXED: Move loss function to device if it has parameters/buffers
    if hasattr(loss_fn, 'to'):
        loss_fn.to(device)

    # FIXED: Don't step scheduler on every iteration - use validation loss instead
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode='min',
        factor=0.5,
        patience=5,  # Increased patience for stability
        verbose=True,
        min_lr=1e-6
    )

    logs = []
    running_loss = 0.0  # For tracking accumulated loss
    optimizer.zero_grad()  # Initialize gradients

    print(f"Starting training for {num_iterations} iterations")
    print(f"Gradient accumulation steps: {accumulation_steps}")
    print(f"Effective batch size: {train_loader.batch_size * accumulation_steps}")

    for step, (inputs, targets) in enumerate(train_loader):
        if step >= num_iterations:
            break

        start_time = time.time()
        model.train()

        inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)

        # Forward pass
        logits = model(inputs)  # [B, C, D, H, W]
        loss, stats = loss_fn(logits, targets)

        # FIXED: Normalize loss by accumulation steps for proper gradients
        normalized_loss = loss / accumulation_steps
        normalized_loss.backward()

        # Accumulate loss for logging
        running_loss += loss.item()

        # FIXED: Only step optimizer after accumulation_steps, not scheduler
        if (step + 1) % accumulation_steps == 0:
            # Optional: Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            optimizer.zero_grad()

            # Calculate average loss over accumulation steps
            avg_loss = running_loss / accumulation_steps
            running_loss = 0.0

            # FIXED: Step scheduler less frequently (every few optimizer steps)
            if (step + 1) % (accumulation_steps * 5) == 0:
                scheduler.step(avg_loss)

        step_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']
        class_metrics = compute_metrics_from_stats(stats)

        log_entry = {
            'step': step,
            'step_time_sec': round(step_time, 3),
            'loss': loss.item(),  # Log actual loss (not normalized)
            'lr': current_lr,
            'class_stats': class_metrics,
            'dice_scores': {f'class_{i}': class_metrics[f'class_{i}']['dice']
                          for i in range(len(class_metrics))}
        }
        logs.append(log_entry)

        # FIXED: More informative console output
        if step % 10 == 0 or (step + 1) % accumulation_steps == 0:  # Print less frequently
            print(f"[{step:04d}] {step_time:.2f}s | Loss: {loss.item():.4f} | LR: {current_lr:.2e}")

            # Show dice scores (more meaningful than TP/FP/FN)
            dice_str = " | ".join([f"C{i}: {class_metrics[f'class_{i}']['dice']:.3f}"
                                  for i in range(len(class_metrics))])
            print(f"         Dice: {dice_str}")

        # FIXED: Save less frequently to avoid I/O overhead
        if step % 50 == 0 or step == num_iterations - 1:
            # Save model checkpoint
            checkpoint = {
                'step': step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': loss.item(),
                'logs': logs
            }
            torch.save(checkpoint, os.path.join(save_dir, f'checkpoint_step_{step}.pth'))

            # Save metrics log
            with open(os.path.join(save_dir, 'training_metrics.json'), 'w') as f:
                json.dump(logs, f, indent=2)

    # FIXED: Handle remaining gradients properly
    if (step + 1) % accumulation_steps != 0:
        # Scale final gradients if partial accumulation
        for param in model.parameters():
            if param.grad is not None:
                param.grad.data.mul_(accumulation_steps / ((step + 1) % accumulation_steps))
        optimizer.step()
        optimizer.zero_grad()

    # Save final model
    final_checkpoint = {
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'final_loss': loss.item(),
        'logs': logs
    }
    torch.save(final_checkpoint, os.path.join(save_dir, 'final_model.pth'))

    print(f"\nTraining completed! Final model saved to {save_dir}")
    return logs

# FIXED: Proper validation function to guide scheduler
def validate_model(model, val_loader, loss_fn, device='cuda'):
    """Validation function to get loss for scheduler"""
    model.eval()
    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss, stats = loss_fn(logits, targets)

            total_loss += loss.item() * inputs.size(0)
            total_samples += inputs.size(0)

    avg_loss = total_loss / total_samples
    return avg_loss




In [ ]:

def compute_normalization_stats(image_dir: str, sample_ids: List[str]) -> Dict[str, np.ndarray]:
    """
    Precompute global min/max for each modality across all training samples.

    Args:
        image_dir: Directory containing training images
        sample_ids: List of sample filenames (without extension)

    Returns:
        Dictionary with 'min' and 'max' arrays (shape: [4] for each modality)
    """
    global_min = np.full(4, np.inf)
    global_max = np.full(4, -np.inf)

    for sample_id in sample_ids:
        img_path = os.path.join(image_dir, f"{sample_id}.nii.gz")
        img = nib.load(img_path).get_fdata()  # [H, W, D, 4]

        for k in range(4):  # For each modality
            channel_data = img[..., k]
            global_min[k] = min(global_min[k], np.min(channel_data))
            global_max[k] = max(global_max[k], np.max(channel_data))

    return {'min': global_min, 'max': global_max}

class BratsNormalizer:
    """
    Normalizes BRATS volumes using precomputed min/max per modality.
    """
    def __init__(self, stats: Dict[str, np.ndarray], eps: float = 1e-4):
        """
        Args:
            stats: Dictionary with 'min' and 'max' arrays (shape: [4])
            eps: Small constant for numerical stability
        """
        self.global_min = torch.from_numpy(stats['min']).float()
        self.global_max = torch.from_numpy(stats['max']).float()
        self.eps = eps

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        """
        Args:
            tensor: Input tensor of shape [C, D, H, W]
        Returns:
            Normalized tensor of same shape
        """
        normalized = torch.zeros_like(tensor)
        for k in range(4):  # For each modality/channel
            normalized[k] = (tensor[k] - self.global_min[k]) /  (self.global_max[k] - self.global_min[k] + self.eps)
        return normalized
class BratsDataset(Dataset):
    def __init__(
        self,
        image_dir: str,
        labels_dir: str,
        transform = None,
        normalization_stat = None,
        cache: bool = False,  # Optional in-memory caching
    ):
        self.image_dir = image_dir
        self.label_dir = labels_dir
        self.transform = transform
        self.cache = cache
        self._cache = {}  # Stores cached samples {idx: (img, label)}

        # Get common filenames (without extensions)
        image_files = {f.split(".")[0] for f in os.listdir(image_dir)}
        label_files = {f.split(".")[0] for f in os.listdir(labels_dir)}
        self.common_filenames = sorted(image_files & label_files)

        # Set up normalization
        self.normalizer = BratsNormalizer(normalization_stats) if normalization_stats else None

    def __len__(self):
        return len(self.common_filenames)

    def __getitem__(self, index):
        if index in self._cache:  # Return cached sample if available
            img_tensor, label_tensor = self._cache[index]
        else:
            # LAZY LOADING: Only load when needed
            file_id = self.common_filenames[index]

            # Load image and label from disk
            img = nib.load(os.path.join(self.image_dir, f"{file_id}.nii.gz")).get_fdata()
            label = nib.load(os.path.join(self.label_dir, f"{file_id}.nii.gz")).get_fdata()

            # Convert to tensors and permute dimensions
            img_tensor = torch.from_numpy(img).float().permute(3, 2, 0, 1)  # [C, D, H, W]
            label_tensor = torch.from_numpy(label).long().permute(2, 0, 1)  # [D, H, W]

            if self.normalizer:
                img_tensor = self.normalizer(img_tensor)

            if self.transform:
                img_tensor = self.transform(img_tensor)

            if self.cache:  # Store in cache if enabled
                self._cache[index] = (img_tensor, label_tensor)

        return img_tensor, label_tensor

In [ ]:
class NormalizedActivation(nn.Module):
    """
    normalization + activation for convolutional layers, dynamically normalizes and applies activation on output feature maps

    mu: learnable parameter, similar to mean, but not neceessarily computed in statistical manner,
            helps in shifting and centering data by adapating to channels inherent data values and spread

    alpha: learnable scale (std_dev) for normalization, passed through softplus to ensure positive values
    beta: learnable scaling quantity that can learn to scale contrasting or emphasize non-linearity based on the values
    gamma: controls threshold of non-linear transformations

    placed after convolutional layer that combines both normalization and activations

    uses 1 sample per batch
    input_shape: (batch, channel, depth, height, width), (1, C, D, H, W)
    output_shape: (batch, channel, depth, height, width), (1, C, D, H, W)
    """

    def __init__(self, channels, eps=1e-2):
        # initializes learnable parameters for each channel
        """
        Args:
            - channels: number of input channels, typically the output channels of convolutions
            - eps: small epsilon added to avoid division by zero if alpha goes to zero
        """

        super().__init__()

        self.channels = channels
        self.mu = self._learnable_paramemters()
        self.gamma = self._learnable_paramemters()
        self.beta = self._learnable_paramemters()
        self.alpha = self._learnable_paramemters()
        self.eps = eps


    def _learnable_paramemters(self):
        # initializes parameters with ones
        return nn.Parameter(torch.ones(self.channels, dtype=torch.float32))


    def forward(self, x):
        """
        applies normalization to the input tensor which is the output of convolution operation

        Args:
            - x: input tensor of shape (batch, channel, depth, height, width)

        returns:
            - transformed tensor of the same shape

        """

        # target shape for broadcasting on channels
        shape = (1, -1, 1, 1, 1)

        # (1, channel, 1, 1, 1)
        mu = self.mu.view(shape)
        beta = self.beta.view(shape)
        gamma = self.gamma.view(shape)

        # applying softplus  to ensure >=0, where eps is added to avoid division by zero
        alpha = F.softplus(self.alpha.view(shape)) + self.eps

        # g(x) = x - learned_center / spread
        gx = (x - mu) / alpha


        # smooth clamping to -gamma, gamma
        gx = torch.tanh(gx/gamma) * gamma

        # applies sigmoid  squashing outputs to (0, 1)
        gx =F.SiLU(beta * gx)

        return gx

In [ ]:
def compute_normalization_stats(image_dir: str, sample_ids: List[str]) -> Dict[str, np.ndarray]:
    """
    Precompute global min/max for each modality with lazy loading to conserve memory.
    """
    global_min = np.full(4, np.inf)
    global_max = np.full(4, -np.inf)

    for sample_id in sample_ids:
        img_path = os.path.join(image_dir, f"{sample_id}.nii.gz")
        img = nib.load(img_path).get_fdata()  # Memory-mapped by default

        for k in range(4):  # For each modality
            channel_data = img[..., k]
            global_min[k] = min(global_min[k], np.nanmin(channel_data))
            global_max[k] = max(global_max[k], np.nanmax(channel_data))

    return {'min': global_min, 'max': global_max}

class BratsNormalizer:
    """
    Normalizes BRATS volumes using precomputed min/max per modality.
    Supports both torch tensors and numpy arrays.
    """
    def __init__(self, stats: Dict[str, np.ndarray], eps: float = 1e-7):
        self.global_min = torch.from_numpy(stats['min']).float()
        self.global_max = torch.from_numpy(stats['max']).float()
        self.eps = eps

    def __call__(self, data: Union[torch.Tensor, np.ndarray]) -> torch.Tensor:
        if isinstance(data, np.ndarray):
            data = torch.from_numpy(data).float()

        normalized = torch.zeros_like(data)
        for k in range(4):
            channel_range = self.global_max[k] - self.global_min[k] + self.eps
            normalized[k] = (data[k] - self.global_min[k]) / channel_range
        return normalized

# ====================== LAZY-LOADING DATASET ========================
class BratsDataset(Dataset):
    def __init__(self,
                 image_dir: str,
                 labels_dir: str,
                 transform = None,
                 normalization_stats = None,
                 cache: bool = False):
        """
        Args:
            cache: If True, keeps loaded samples in memory
        """
        self.image_dir = image_dir
        self.label_dir = labels_dir
        self.transform = transform
        self.cache = cache
        self._cache = {}  # {index: (image_tensor, label_tensor)}

        # Find matching image/label pairs
        image_files = {f.split('.')[0] for f in os.listdir(image_dir) if f.endswith('.nii.gz')}
        label_files = {f.split('.')[0] for f in os.listdir(labels_dir) if f.endswith('.nii.gz')}
        self.common_filenames = sorted(image_files & label_files)

        # Initialize normalizer if stats provided
        self.normalizer = BratsNormalizer(normalization_stats) if normalization_stats else None

    def __len__(self):
        return len(self.common_filenames)

    def _load_sample(self, index):
        """Internal method for lazy loading"""
        file_id = self.common_filenames[index]

        # Load using nibabel (memory-mapped by default)
        img = nib.load(os.path.join(self.image_dir, f"{file_id}.nii.gz")).get_fdata()
        label = nib.load(os.path.join(self.label_dir, f"{file_id}.nii.gz")).get_fdata()

        # Convert to tensors with correct dimensions
        img_tensor = torch.from_numpy(img).float().permute(3, 2, 0, 1)  # [C, D, H, W]
        label_tensor = torch.from_numpy(label).long().permute(2, 0, 1)   # [D, H, W]

        return img_tensor, label_tensor

    def __getitem__(self, index):
        if index in self._cache:
            img, label = self._cache[index]
        else:
            img, label = self._load_sample(index)
            if self.cache:
                self._cache[index] = (img, label)

        # Apply transforms and normalization
        if self.normalizer:
            img = self.normalizer(img)
        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
temp_dataset = BratsDataset(bt_train_images_dir, bt_train_labels_dir)

# Step 2: Compute normalization statistics (this may take a while)
norm_stats = compute_normalization_stats(
    bt_train_images_dir,
    temp_dataset.common_filenames
)

# Optional: Save stats for future use
np.save(os.path.join(braintumor_datadir, "norm_stats.npy"), norm_stats)

# Step 3: Create final dataset with lazy loading + normalization
bt_train_dataset = BratsDataset(
    image_dir=bt_train_images_dir,
    labels_dir=bt_train_labels_dir,
    normalization_stats=norm_stats,
    cache=True  # Enable caching for faster epoch-to-epoch loading
)

# Step 4: Create DataLoader
bt_dataloader = DataLoader(
    bt_train_dataset,
    batch_size=1,           # Small batch sizes for 3D data
    shuffle=True,
    num_workers=0,          # Reduced workers to potentially save memory
    pin_memory=False,        # Disabled pin_memory
    persistent_workers=False # Disabled persistent_workers
)

# Test iteration
for batch_idx, (images, labels) in enumerate(bt_dataloader):
    print(f"Batch {batch_idx}: Images {images.shape}, Labels {labels.shape}")
    if batch_idx == 1:  # Just show first 3 batches
        break

In [ ]:
def conv_norm_act(in_channels, out_channels, kernel=3, padding=1, stride=1):
    return nn.Sequential(
        nn.Conv3d(in_channels, out_channels, kernel_size=kernel, padding=padding, stride=stride),
        nn.GroupNorm(out_channels//4, num_channels=out_channels),
        nn.SiLU()
    )

def conv_norm_act_upsample(in_channels, out_channels, target_size, kernel=3, padding=1, stride=1):
    return nn.Sequential(
        nn.Upsample(size=target_size, mode='trilinear', align_corners=True),
        nn.Conv3d(in_channels, out_channels, kernel_size=kernel, padding=padding, stride=stride),
        nn.GroupNorm(out_channels//4, num_channels=out_channels),
        nn.SiLU()
    )


class UNet3D(nn.Module):

    def __init__(self, in_channels=4, out_channels=[8, 16, 32, 64, 128]):
        super().__init__()

        # input_size: (1, 4, 155, 240, 240)
        self.encoder_list = nn.ModuleList([
            conv_norm_act(in_channels, out_channels[0]), # e0: (1, 4, 155, 240, 240) -> (1, 8, 155, 240, 240), skip0
            conv_norm_act(out_channels[0], out_channels[1], stride=2), # e1: (1, 8, 155, 240, 240) -> (1, 16, 78, 120, 120), skip1
            conv_norm_act(out_channels[1], out_channels[2], stride=2), # e2: (1, 16, 78, 120, 120) -> (1, 32, 39, 60, 60), skip2
            conv_norm_act(out_channels[2], out_channels[3], stride=2), # e3: (1, 32, 39, 60, 60) -> (1, 64, 20, 30, 30), skip3
            conv_norm_act(out_channels[3], out_channels[4], stride=2), # e4: (1, 64, 20, 30, 30) -> (1, 128, 10, 15, 15), skip4
        ])

        self.bottleneck = conv_norm_act(out_channels[4], out_channels[4] * 2) #bottlneck: (1, 128, 10, 15, 15) -> (1, 256, 10, 15, 15)

        self.decoder_list = nn.ModuleList([
            conv_norm_act(256, 128), # d0:(1, 256, 10, 15, 15) -> (1, 128, 10, 15, 15)
            conv_norm_act_upsample(256, 64, target_size=(20, 30, 30)), # d1: (1, 256, 10, 15, 15) -> (1, 64, 20, 30, 30)
            conv_norm_act_upsample(128, 32, target_size=(39, 60, 60)), # d2: (1, 128, 20, 30, 30) -> (1, 32, 39, 60, 60)
            conv_norm_act_upsample(64, 16, target_size=(78, 120, 120)), # d3: (1, 64, 39, 60, 60) -> (1, 16, 78, 120, 120)
            conv_norm_act_upsample(32, 8, target_size=(155, 240, 240)), # d4: (1, 32, 78, 120, 120) -> (1, 8, 155, 240, 240)
        ])

        self.final_conv = nn.Conv3d(16, 4, kernel_size=3, padding=1)


    def forward(self, x: torch.tensor):
        skips = []

        for encoder in self.encoder_list:
            x = encoder(x)
            skips.append(x)

        x = self.bottleneck(x)

        skips = skips[::-1]

        for decoder, skip in zip(self.decoder_list, skips):
            x = decoder(x)
            x = torch.cat([x, skip], dim=1)


        x = self.final_conv(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedDiceLoss(nn.Module):
    def __init__(self, num_classes=4,
                 false_positive_weights=None, false_negative_weights=None,
                 class_importance_weights=None, epsilon=1e-6):
        """
        Args:
            num_classes (int): number of segmentation classes
            false_positive_weights (Tensor): shape [C], λ^FP_c per class
            false_negative_weights (Tensor): shape [C], λ^FN_c per class
            class_importance_weights (Tensor, optional): shape [C], γ_c or w_c
            epsilon (float): smoothing to avoid division by zero
        """
        super().__init__()
        self.num_classes = num_classes
        # Register weights as buffers so they are moved to the device with the module
        self.register_buffer('fp_weights', false_positive_weights)
        self.register_buffer('fn_weights', false_negative_weights)
        self.register_buffer('class_weights', class_importance_weights)
        self.epsilon = epsilon

    def forward(self, logits, targets):
        """
        Args:
            logits: Tensor of shape [B, C, D, H, W]
            targets: Tensor of shape [B, D, H, W] with values in 0,...,C-1
        Returns:
            Scalar loss value
        """
        B, C, *spatial = logits.shape
        probs = F.softmax(logits, dim=1)  # Step 1: Softmax
        targets_onehot = F.one_hot(targets, num_classes=C).permute(0, 4, 1, 2, 3).float()  # Step 2: One-hot

        # Step 3: Compute TP, FP, FN
        TP = (probs * targets_onehot).sum(dim=[0, 2, 3, 4])
        FP = (probs * (1 - targets_onehot)).sum(dim=[0, 2, 3, 4])
        FN = ((1 - probs) * targets_onehot).sum(dim=[0, 2, 3, 4])

        # Step 4: Compute class frequency weights if not provided
        if self.class_weights is None:
            class_counts = targets_onehot.sum(dim=[0, 2, 3, 4])
            # Ensure class_weights are on the same device as other tensors
            class_weights = (1.0 / (class_counts + self.epsilon)).to(logits.device)
            class_weights = class_weights / class_weights.sum()
        else:
            class_weights = self.class_weights


        # Step 5-6: Compute Tversky loss per class
        numerator = TP
        denominator = TP + self.fp_weights * FP + self.fn_weights * FN + self.epsilon
        class_losses = 1.0 - numerator / denominator  # [C]

        # Step 7: Weighted sum of losses
        total_loss = (class_weights * class_losses).sum()

        stats = {
        'TP': TP.detach().cpu(),
        'FP': FP.detach().cpu(),
        'FN': FN.detach().cpu()
    }

        return total_loss, stats

In [ ]:
def compute_metrics_from_stats(stats):
    """
    Args:
        stats: dict with keys 'TP', 'FP', 'FN' as tensors of shape [C]
    Returns:
        dict: class_idx -> precision, recall, etc.
    """
    TP = stats['TP']
    FP = stats['FP']
    FN = stats['FN']

    metrics = {}
    for c in range(len(TP)):
        tp = TP[c].item()
        fp = FP[c].item()
        fn = FN[c].item()

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)

        metrics[c] = {
            'TP': tp,
            'FP': fp,
            'FN': fn,
            'precision': precision,
            'recall': recall
        }

    return metrics


In [ ]:
model_w_dicev1 = UNet3D()
optimizer_w_dicev1 = AdamW(model_w_dicev1.parameters(), lr=1e-4, weight_decay=1e-3)
num_classes = 4
fp_weights = torch.tensor([1.0] * num_classes)  # λ_FP
fn_weights = torch.tensor([1.0] * num_classes)  # λ_FN
class_weights = None  # or provide custom weights as tensor of shape [C]

loss_dice_v1 = WeightedDiceLoss(fp_weights=torch.tensor([0.1, 1.0, 1.0, 1.0]),
                 fn_weights=torch.tensor([1.0, 1.0, 1.0, 1.0]),
                 class_weights=torch.tensor([0.2, 1.0, 1.0, 1.0]))



In [ ]:
def trainV1(model, train_loader, loss_fn, optimizer, num_iterations, save_dir,
          device='cuda', early_stop_patience=10, accumulation_steps=4): # Added accumulation_steps

    os.makedirs(save_dir, exist_ok=True)
    model.to(device)
    loss_fn.to(device)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode='min',
        factor=0.5,
        patience=3,
        verbose=True,
        min_lr=1e-6
    )

    logs = []
    optimizer.zero_grad() # Initialize gradients

    for step, (inputs, targets) in enumerate(train_loader):
        if step >= num_iterations:
            break

        start_time = time.time()

        model.train()
        inputs, targets = inputs.to(device), targets.to(device)

        logits = model(inputs)  # [B, C, D, H, W]
        loss, stats = loss_fn(logits, targets)

        # Normalize loss by accumulation steps
        loss = loss / accumulation_steps

        loss.backward()

        # Perform optimizer step and clear gradients only after accumulation_steps
        if (step + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step(loss) # Step scheduler after optimizer update

        step_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']
        class_metrics = compute_metrics_from_stats(stats)

        log_entry = {
            'step': step,
            'step_time_sec': round(step_time, 3),
            'loss': loss.item() * accumulation_steps, # Log the un-normalized loss
            'lr': current_lr,
            'class_stats': class_metrics
        }
        logs.append(log_entry)

        # Console Output
        print(f"[{step:04d}] {step_time:.2f}s | Loss: {log_entry['loss']:.4f} | LR: {current_lr:.2e}")
        for cls_id, cm in class_metrics.items():
            print(f"  Class {cls_id}: TP={cm['TP']}  FP={cm['FP']}  FN={cm['FN']}  "
                  f"Precision={cm['precision']:.3f}  Recall={cm['recall']:.3f}")

        # Save model + metrics log (optional, might want to save less frequently)
        # torch.save(model.state_dict(), os.path.join(save_dir, f'model_step{step}.pth'))
        with open(os.path.join(save_dir, 'metrics.json'), 'w') as f:
            json.dump(logs, f, indent=2)

    # Perform a final optimizer step if there are remaining gradients
    if (step + 1) % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

    return logs

In [ ]:
trainV1(model_w_dicev1, bt_dataloader, loss_dice_v1, optimizer_w_dicev1, 50, "/dice_v1")

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[0000] 0.59s | Loss: 0.9998 | LR: 1.00e-04
  Class 0: TP=1743823.0  FP=16873.88671875  FN=7086192.0  Precision=0.990  Recall=0.197
  Class 1: TP=51396.5625  FP=3080449.75  FN=41885.4375  Precision=0.016  Recall=0.551
  Class 2: TP=171.66055297851562  FP=1389311.0  FN=3643.339599609375  Precision=0.000  Recall=0.045
  Class 3: TP=244.04000854492188  FP=2645729.75  FN=643.9599609375  Precision=0.000  Recall=0.275
[0001] 0.77s | Loss: 0.9960 | LR: 1.00e-04
  Class 0: TP=1730081.0  FP=32815.7265625  FN=7058662.0  Precision=0.981  Recall=0.197
  Class 1: TP=32137.59375  FP=3111982.5  FN=41603.40625  Precision=0.010  Recall=0.436
  Class 2: TP=6276.345703125  FP=1286537.375  FN=39562.65625  Precision=0.005  Recall=0.137
  Class 3: TP=4456.66650390625  FP=2723713.0  FN=15220.333984375  Precision=0.002  Recall=0.226
[0002] 0.57s | Loss: 0.9971 | LR: 1.00e-04
  Class 0: TP=1777141.625  FP=27616.98828125  FN=6971550.5  Precision=0.985  Recall=0.203
  Class 1: TP=80432.953125  FP=3004401.75  FN=5

RuntimeError: DataLoader worker (pid 5319) is killed by signal: Killed. 

In [ ]:
def train(model, train_loader, loss_fn, optimizer, num_iterations, save_dir,
          device='cuda', early_stop_patience=10):

    os.makedirs(save_dir, exist_ok=True)
    model.to(device)
    loss_fn.to(device)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode='min',
        factor=0.5,
        patience=3,
        verbose=True,
        min_lr=1e-6
    )

    logs = []

    for step, (inputs, targets) in enumerate(train_loader):
        if step >= num_iterations:
            break

        start_time = time.time()

        model.train()
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model(inputs)  # [B, C, D, H, W]
        loss, stats = loss_fn(logits, targets)

        loss.backward()
        #clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step(loss)

        step_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']
        class_metrics = compute_metrics_from_stats(stats)

        log_entry = {
            'step': step,
            'step_time_sec': round(step_time, 3),
            'loss': loss.item(),
            'lr': current_lr,
            'class_stats': class_metrics
        }
        logs.append(log_entry)

        # Console Output
        print(f"[{step:04d}] {step_time:.2f}s | Loss: {loss.item():.4f} | LR: {current_lr:.2e}")
        for cls_id, cm in class_metrics.items():
            print(f"  Class {cls_id}: TP={cm['TP']}  FP={cm['FP']}  FN={cm['FN']}  "
                  f"Precision={cm['precision']:.3f}  Recall={cm['recall']:.3f}")


    return logs


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# DIAGNOSIS: Your current training issues
print("=== TRAINING ISSUES IDENTIFIED ===")
print("❌ Loss stuck at ~0.99 (not learning)")
print("❌ Class 0 (background): 99.9% precision → model predicts mostly background")
print("❌ Classes 1,2,3 (tumors): Near 0% Dice → barely detecting tumors")
print("❌ LR dropping too aggressively (1e-4 → 4.9e-5 in 150 steps)")
print("❌ High recall but low precision on tumor classes → many false positives")
print("\n=== ROOT CAUSES ===")
print("1. Extreme class imbalance (90% background)")
print("2. Loss function weights may need adjustment")
print("3. Learning rate schedule too aggressive")
print("4. Model may need warmup period")
print("5. Possibly need different sampling strategy")

# SOLUTION 1: Enhanced loss function with better class balancing
class AdaptiveWeightedDiceLoss(nn.Module):
    """Enhanced version of your loss with adaptive weighting"""
    def __init__(self, num_classes=4, epsilon=1e-6,
                 focal_gamma=2.0, adaptive_weights=True):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.focal_gamma = focal_gamma
        self.adaptive_weights = adaptive_weights

        # More aggressive weights for tumor classes
        self.register_buffer('fp_weights', torch.tensor([0.1, 1.0, 2.0, 3.0]))  # Lower FP penalty for background
        self.register_buffer('fn_weights', torch.tensor([0.1, 3.0, 4.0, 5.0]))  # Much higher FN penalty for tumors

    def forward(self, logits, targets):
        B, C = logits.shape[:2]

        # Softmax with temperature to prevent overconfident predictions
        probs = F.softmax(logits / 1.2, dim=1)  # Temperature = 1.2

        # One-hot encoding
        targets_onehot = F.one_hot(targets.long(), num_classes=4).float()
        targets_onehot = targets_onehot.permute(0, 4, 1, 2, 3)

        # Compute TP, FP, FN
        TP = (probs * targets_onehot).sum(dim=(0, 2, 3, 4))
        FP = (probs * (1 - targets_onehot)).sum(dim=(0, 2, 3, 4))
        FN = ((1 - probs) * targets_onehot).sum(dim=(0, 2, 3, 4))

        # Adaptive class weights based on frequency
        if self.adaptive_weights:
            class_counts = targets_onehot.sum(dim=(0, 2, 3, 4))
            total_pixels = targets_onehot.sum()

            # Inverse frequency weighting with smoothing
            class_weights = total_pixels / (self.num_classes * class_counts + self.epsilon)
            class_weights = class_weights / class_weights.sum()  # Normalize

            # Boost tumor class weights even more
            class_weights[1:] *= 2.0  # Double weight for tumor classes
            class_weights = class_weights / class_weights.sum()  # Re-normalize
        else:
            class_weights = torch.ones(4, device=logits.device) / 4

        # Focal weighting to focus on hard examples
        dice_scores = (2 * TP + self.epsilon) / (2 * TP + FP + FN + self.epsilon)
        focal_weights = (1 - dice_scores) ** self.focal_gamma

        # Tversky loss computation
        numerator = TP
        denominator = TP + self.fp_weights * FP + self.fn_weights * FN + self.epsilon
        tversky_scores = numerator / denominator

        # Combined loss with focal weighting
        class_losses = (1.0 - tversky_scores) * focal_weights
        total_loss = (class_weights * class_losses).sum()

        # Enhanced stats
        stats = {
            'TP': TP.detach().cpu(),
            'FP': FP.detach().cpu(),
            'FN': FN.detach().cpu(),
            'class_losses': class_losses.detach().cpu(),
            'class_weights': class_weights.detach().cpu(),
            'dice_scores': dice_scores.detach().cpu(),
            'focal_weights': focal_weights.detach().cpu()
        }

        return total_loss, stats

# SOLUTION 2: Better learning rate schedule
def create_better_scheduler(optimizer):
    """More suitable LR schedule for medical segmentation"""
    return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=50,      # Restart every 50 steps initially
        T_mult=2,    # Double the period after each restart
        eta_min=1e-6 # Minimum learning rate
    )

# SOLUTION 3: Warmup training function
def warmup_training_step(model, optimizer, warmup_steps=20):
    """Gradual warmup for the first few steps"""
    def set_lr(step):
        if step < warmup_steps:
            lr_scale = (step + 1) / warmup_steps
            for param_group in optimizer.param_groups:
                param_group['lr'] = param_group['lr'] * lr_scale
    return set_lr

# SOLUTION 4: Enhanced training function with fixes
def trainV3_fixed(model, train_loader, num_iterations, save_dir,
                  device='cuda', accumulation_steps=4, mixed_precision=True):
    """Fixed training function addressing your specific issues"""

    os.makedirs(save_dir, exist_ok=True)
    model.to(device)

    # Enhanced loss function
    loss_fn = AdaptiveWeightedDiceLoss(
        num_classes=4,
        focal_gamma=2.0,
        adaptive_weights=True
    ).to(device)

    # Better optimizer settings for medical data
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-4,  # Slightly higher initial LR
        weight_decay=1e-4,  # Stronger regularization
        betas=(0.9, 0.999)
    )

    # Better scheduler
    scheduler = create_better_scheduler(optimizer)
    warmup_fn = warmup_training_step(model, optimizer, warmup_steps=20)

    # Mixed precision
    scaler = torch.cuda.amp.GradScaler() if mixed_precision else None

    logs = []
    optimizer.zero_grad()

    print("=== ENHANCED TRAINING STARTED ===")
    print(f"Enhanced loss: AdaptiveWeightedDiceLoss")
    print(f"Better scheduler: CosineAnnealingWarmRestarts")
    print(f"Warmup: 20 steps")
    print(f"Initial LR: 2e-4")

    for step, (inputs, targets) in enumerate(train_loader):
        if step >= num_iterations:
            break

        model.train()

        # Warmup LR for first 20 steps
        if step < 20:
            warmup_fn(step)

        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        # Forward pass with mixed precision
        if mixed_precision:
            with torch.cuda.amp.autocast():
                logits = model(inputs)
                loss, stats = loss_fn(logits, targets)
                loss = loss / accumulation_steps
        else:
            logits = model(inputs)
            loss, stats = loss_fn(logits, targets)
            loss = loss / accumulation_steps

        # Backward pass
        if mixed_precision:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # Optimizer step with gradient clipping
        if (step + 1) % accumulation_steps == 0:
            if mixed_precision:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)  # Tighter clipping
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()

            optimizer.zero_grad()
            scheduler.step()  # Step every optimizer update

        # Enhanced logging
        if step % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']

            # Compute meaningful metrics
            dice_scores = stats['dice_scores']
            mean_tumor_dice = dice_scores[1:].mean().item()  # Average of tumor classes

            print(f"[{step:04d}] Loss: {loss.item() * accumulation_steps:.4f} | "
                  f"LR: {current_lr:.2e} | Mean Tumor Dice: {mean_tumor_dice:.3f}")

            if step % 50 == 0:
                print(f"  Background Dice: {dice_scores[0]:.3f}")
                print(f"  Tumor Classes Dice: {dice_scores[1:]}")
                print(f"  Class weights: {stats['class_weights']}")
                print(f"  Focal weights: {stats['focal_weights']}")

        # Memory cleanup
        del inputs, targets, logits
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return logs

# SOLUTION 5: Data sampling strategy for imbalanced data
class BalancedBatchSampler:
    """Sampler that ensures each batch has both background and tumor samples"""
    def __init__(self, dataset, batch_size=2):
        self.dataset = dataset
        self.batch_size = batch_size

    def __iter__(self):
        # This is a simplified example - you'd implement based on your dataset
        indices = list(range(len(self.dataset)))
        np.random.shuffle(indices)

        for i in range(0, len(indices), self.batch_size):
            yield indices[i:i + self.batch_size]

    def __len__(self):
        return len(self.dataset) // self.batch_size

# SOLUTION 6: Quick diagnostic function
def diagnose_model_predictions(model, sample_batch, device='cuda'):
    """Quick diagnosis of what the model is predicting"""
    model.eval()

    with torch.no_grad():
        inputs, targets = sample_batch
        inputs = inputs.to(device)

        logits = model(inputs)
        probs = F.softmax(logits, dim=1)
        predictions = torch.argmax(probs, dim=1)

        # Count predictions per class
        for i in range(4):
            pred_count = (predictions == i).sum().item()
            target_count = (targets == i).sum().item()
            pred_pct = pred_count / predictions.numel() * 100
            target_pct = target_count / targets.numel() * 100

            print(f"Class {i}: Predicted {pred_pct:.1f}% | Actual {target_pct:.1f}%")

        # Check if model is too conservative
        max_probs = torch.max(probs, dim=1)[0]
        avg_confidence = max_probs.mean().item()
        print(f"Average prediction confidence: {avg_confidence:.3f}")

        if avg_confidence > 0.95:
            print("⚠️  Model is overconfident - consider temperature scaling")

        return predictions, probs
diagnose_model_predictions(optimized_model, create_memory_efficient_dataloader(bt_dataset))
print("\n=== RECOMMENDED ACTIONS ===")
print("1. Replace your loss with AdaptiveWeightedDiceLoss")
print("2. Use the enhanced training function trainV3_fixed")
print("3. Monitor 'Mean Tumor Dice' instead of just loss")
print("4. If still struggling, try smaller patches or data augmentation")
print("5. Consider using the diagnostic function to check predictions")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

print("=== DIAGNOSIS OF CURRENT MODEL STATE ===")
print("✅ GOOD: Model is now predicting tumor classes (was 0% before)")
print("✅ GOOD: Low confidence (0.414) means it's not overconfident")
print("❌ ISSUE: Severe overcorrection - predicting 56% tumors vs 1.1% actual")
print("❌ ISSUE: Too many false positives in classes 2 & 3")
print("\n=== ROOT CAUSE ===")
print("The enhanced loss weights are TOO aggressive - need calibration")

# SOLUTION 1: Calibrated loss with gentler weighting
class CalibratedWeightedDiceLoss(nn.Module):
    """Calibrated version with balanced tumor detection"""
    def __init__(self, num_classes=4, epsilon=1e-6,
                 stage='initial'):  # 'initial', 'intermediate', 'fine'
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.stage = stage

        # Progressive weight adjustment based on training stage
        if stage == 'initial':
            # Gentle start - just slightly favor tumors
            self.register_buffer('fp_weights', torch.tensor([0.5, 1.2, 1.5, 1.8]))
            self.register_buffer('fn_weights', torch.tensor([0.3, 1.8, 2.2, 2.5]))
        elif stage == 'intermediate':
            # Medium aggression once model starts learning
            self.register_buffer('fp_weights', torch.tensor([0.3, 1.0, 1.3, 1.6]))
            self.register_buffer('fn_weights', torch.tensor([0.2, 2.0, 2.5, 3.0]))
        else:  # fine
            # Conservative fine-tuning to reduce false positives
            self.register_buffer('fp_weights', torch.tensor([0.2, 0.8, 1.0, 1.2]))
            self.register_buffer('fn_weights', torch.tensor([0.1, 1.5, 2.0, 2.5]))

    def forward(self, logits, targets):
        B, C = logits.shape[:2]

        # Gentle temperature scaling
        probs = F.softmax(logits / 1.1, dim=1)

        # One-hot encoding
        targets_onehot = F.one_hot(targets.long(), num_classes=4).float()
        targets_onehot = targets_onehot.permute(0, 4, 1, 2, 3)

        # Compute TP, FP, FN
        TP = (probs * targets_onehot).sum(dim=(0, 2, 3, 4))
        FP = (probs * (1 - targets_onehot)).sum(dim=(0, 2, 3, 4))
        FN = ((1 - probs) * targets_onehot).sum(dim=(0, 2, 3, 4))

        # Moderate class weighting (less aggressive than before)
        class_counts = targets_onehot.sum(dim=(0, 2, 3, 4))
        total_pixels = targets_onehot.sum()

        # Sqrt instead of linear inverse weighting (gentler)
        class_weights = torch.sqrt(total_pixels / (self.num_classes * class_counts + self.epsilon))
        class_weights = class_weights / class_weights.sum()

        # Cap the maximum weight to prevent extreme imbalance
        class_weights = torch.clamp(class_weights, max=0.7)
        class_weights = class_weights / class_weights.sum()

        # Tversky loss
        numerator = TP
        denominator = TP + self.fp_weights * FP + self.fn_weights * FN + self.epsilon
        tversky_scores = numerator / denominator
        class_losses = 1.0 - tversky_scores

        total_loss = (class_weights * class_losses).sum()

        # Detailed stats for monitoring
        precision = TP / (TP + FP + self.epsilon)
        recall = TP / (TP + FN + self.epsilon)
        dice_scores = (2 * TP) / (2 * TP + FP + FN + self.epsilon)

        stats = {
            'TP': TP.detach().cpu(),
            'FP': FP.detach().cpu(),
            'FN': FN.detach().cpu(),
            'precision': precision.detach().cpu(),
            'recall': recall.detach().cpu(),
            'dice_scores': dice_scores.detach().cpu(),
            'class_weights': class_weights.detach().cpu(),
            'class_losses': class_losses.detach().cpu()
        }

        return total_loss, stats

# SOLUTION 2: Progressive training strategy
class ProgressiveTrainer:
    """Train in stages with different loss configurations"""

    def __init__(self, model, device='cuda'):
        self.model = model
        self.device = device
        self.stage = 'initial'

    def get_loss_and_optimizer(self):
        """Get loss and optimizer for current stage"""
        loss_fn = CalibratedWeightedDiceLoss(stage=self.stage).to(self.device)

        if self.stage == 'initial':
            # Higher LR for initial learning
            optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=1e-4,
                weight_decay=1e-5
            )
        elif self.stage == 'intermediate':
            # Medium LR for refinement
            optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=5e-5,
                weight_decay=1e-4
            )
        else:  # fine
            # Low LR for fine-tuning
            optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=1e-5,
                weight_decay=1e-4
            )

        return loss_fn, optimizer

    def should_advance_stage(self, logs, min_steps=50):
        """Determine if ready to advance to next stage"""
        if len(logs) < min_steps:
            return False

        recent_logs = logs[-10:]  # Last 10 steps

        if self.stage == 'initial':
            # Advance when tumor classes start getting detected
            avg_tumor_dice = np.mean([
                log['class_stats'].get('class_1', {}).get('dice', 0) +
                log['class_stats'].get('class_2', {}).get('dice', 0) +
                log['class_stats'].get('class_3', {}).get('dice', 0)
                for log in recent_logs if 'class_stats' in log
            ]) / 3
            return avg_tumor_dice > 0.05  # When any tumor detection starts

        elif self.stage == 'intermediate':
            # Advance when dice scores stabilize but FP still high
            tumor_dice_scores = []
            for log in recent_logs:
                if 'class_stats' in log:
                    for cls in ['class_1', 'class_2', 'class_3']:
                        if cls in log['class_stats']:
                            tumor_dice_scores.append(log['class_stats'][cls].get('dice', 0))

            if tumor_dice_scores:
                avg_dice = np.mean(tumor_dice_scores)
                return avg_dice > 0.15  # When decent tumor detection achieved

        return False

    def advance_stage(self):
        """Move to next training stage"""
        if self.stage == 'initial':
            self.stage = 'intermediate'
            print("🔄 ADVANCING TO INTERMEDIATE STAGE")
            print("   → Focusing on precision improvement")
        elif self.stage == 'intermediate':
            self.stage = 'fine'
            print("🔄 ADVANCING TO FINE-TUNING STAGE")
            print("   → Reducing false positives")

# SOLUTION 3: Enhanced training with progressive strategy
def trainV4_progressive(model, train_loader, num_iterations, save_dir,
                       device='cuda', accumulation_steps=4):
    """Progressive training that adapts loss weights based on model performance"""

    import os
    os.makedirs(save_dir, exist_ok=True)

    trainer = ProgressiveTrainer(model, device)
    loss_fn, optimizer = trainer.get_loss_and_optimizer()

    # Gentle scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=100, T_mult=1, eta_min=1e-6
    )

    logs = []
    scaler = torch.cuda.amp.GradScaler()
    optimizer.zero_grad()

    print(f"=== PROGRESSIVE TRAINING STARTED ===")
    print(f"Stage: {trainer.stage}")
    print(f"Initial LR: {optimizer.param_groups[0]['lr']:.2e}")

    for step, (inputs, targets) in enumerate(train_loader):
        if step >= num_iterations:
            break

        # Check if we should advance training stage
        if trainer.should_advance_stage(logs, min_steps=50):
            trainer.advance_stage()
            loss_fn, optimizer = trainer.get_loss_and_optimizer()
            scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=100, T_mult=1, eta_min=1e-6
            )

        model.train()
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        # Forward pass
        with torch.cuda.amp.autocast():
            logits = model(inputs)
            loss, stats = loss_fn(logits, targets)
            loss = loss / accumulation_steps

        # Backward pass
        scaler.scale(loss).backward()

        # Optimizer step
        if (step + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        # Enhanced logging with balance monitoring
        if step % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            dice_scores = stats['dice_scores']
            precision_scores = stats['precision']
            recall_scores = stats['recall']

            # Calculate key metrics
            mean_tumor_dice = dice_scores[1:].mean().item()
            mean_tumor_precision = precision_scores[1:].mean().item()
            mean_tumor_recall = recall_scores[1:].mean().item()

            log_entry = {
                'step': step,
                'stage': trainer.stage,
                'loss': loss.item() * accumulation_steps,
                'lr': current_lr,
                'mean_tumor_dice': mean_tumor_dice,
                'mean_tumor_precision': mean_tumor_precision,
                'mean_tumor_recall': mean_tumor_recall,
                'dice_scores': dice_scores.tolist(),
                'class_stats': {
                    f'class_{i}': {
                        'dice': float(dice_scores[i]),
                        'precision': float(precision_scores[i]),
                        'recall': float(recall_scores[i])
                    } for i in range(4)
                }
            }
            logs.append(log_entry)

            print(f"[{step:04d}] Stage: {trainer.stage} | Loss: {log_entry['loss']:.4f} | "
                  f"LR: {current_lr:.2e}")
            print(f"        Tumor→ Dice:{mean_tumor_dice:.3f} | "
                  f"Prec:{mean_tumor_precision:.3f} | Rec:{mean_tumor_recall:.3f}")

            if step % 50 == 0:
                print(f"  Class Dice: {dice_scores.tolist()}")
                print(f"  Class Precision: {precision_scores.tolist()}")
                print(f"  Class Recall: {recall_scores.tolist()}")

        # Memory cleanup
        del inputs, targets, logits
        torch.cuda.empty_cache()

    return logs

# SOLUTION 4: Quick fix for your current model
def quick_recalibration(model, sample_data, device='cuda'):
    """Quick recalibration to reduce false positives"""

    print("=== QUICK RECALIBRATION ===")

    # Use gentler loss for immediate improvement
    calibrated_loss = CalibratedWeightedDiceLoss(stage='fine').to(device)

    # Lower learning rate for fine adjustment
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)

    model.train()

    # Run a few calibration steps
    for i in range(10):
        inputs, targets = sample_data
        inputs, targets = inputs.to(device), targets.to(device)

        with torch.cuda.amp.autocast():
            logits = model(inputs)
            loss, stats = calibrated_loss(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if i % 2 == 0:
            dice = stats['dice_scores']
            print(f"Calibration step {i}: Dice = {dice.tolist()}")

    print("✅ Quick recalibration complete")
    return model

print("\n=== RECOMMENDED IMMEDIATE ACTIONS ===")
print("1. Switch to CalibratedWeightedDiceLoss with stage='fine'")
print("2. Lower your learning rate to 1e-5")
print("3. Use the progressive training approach")
print("4. Monitor precision/recall balance, not just dice")
print("5. Your model IS learning - just needs calibration!")

print("\n=== EXPECTED IMPROVEMENTS ===")
print("• Tumor predictions should drop from 56% to ~5-10%")
print("• Precision should improve significantly")
print("• Dice scores should become more balanced")
print("• Less false positives in classes 2 & 3")